# Understanding InsurEnv

In this notebook, we will delve into the `InsurEnv` environment, focusing on understanding how it works step by step, and particularly analyzing the rewards function.


In [1]:

import pandas as pd
import gymnasium as gym
import matplotlib.pyplot as plt
import gym_insurance

from gym_insurance.datasets.loader import load_psr_data
data = load_psr_data("PSR - 2016 a 2021")
data

c:\repos\gym-insurance\gym_insurance\datasets\loader.py:15: DtypeWarning: Columns (2,11,15,23,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='latin-1', sep = ';', skiprows=[526])


KeyError: "None of [Index(['id_proposta'], dtype='object')] are in the [index]"

In [ ]:
data

,NM_RAZAO_SOCIAL,CD_PROCESSO_SUSEP,NR_PROPOSTA,ID_PROPOSTA,DT_PROPOSTA,DT_INICIO_VIGENCIA,DT_FIM_VIGENCIA,NM_SEGURADO,NR_DOCUMENTO_SEGURADO,NM_MUNICIPIO_PROPRIEDADE,...,VL_LIMITE_GARANTIA,VL_PREMIO_LIQUIDO,PE_TAXA,VL_SUBVENCAO_FEDERAL,NR_APOLICE,DT_APOLICE,ANO_APOLICE,CD_GEOCMU,VALOR_INDENIZAÇÃO,EVENTO_PREPONDERANTE
0,Allianz Seguros S.A,15414000340200569,2542590,596254,29/11/2007,23/07/2016,23/07/2016,BRUNO AFONSO LUDWIG,***34538801,Rancharia,...,9680,"677,6","0,0700","338,8",1362,12/12/2007,2007.0,3542206.0,NaN,NaN
1,Allianz Seguros S.A,15414000340200569,2540491,596255,29/11/2007,23/07/2016,23/07/2016,BENEDITO ANDRADE DE OLIVEIRA,***45599831,Rancharia,...,"61034,82","4272,44","0,0700","2136,22",1366,12/12/2007,2007.0,3542206.0,NaN,NaN
2,Allianz Seguros S.A,15414000340200569,2517660,596256,21/11/2007,23/07/2016,23/07/2016,BENEDITO ANDRADE DE OLIVEIRA,***45599831,Maracaí,...,"14247,87","1282,31","0,0900","641,16",0001183,05/12/2007,2007.0,3528809.0,NaN,NaN
3,Allianz Seguros S.A,15414000340200569,2520265,596257,22/11/2007,23/07/2016,23/07/2016,BENEDITO ANDRADE DE OLIVEIRA,***45599831,Maracaí,...,"35748,24","2502,38","0,0700","1251,19",1184,05/12/2007,2007.0,3528809.0,NaN,NaN
4,Allianz Seguros S.A,15414000340200569,2516361,596258,21/11/2007,23/07/2016,23/07/2016,ERVIN NEUMANN,***55934846,Maracaí,...,"44153,28","3090,73","0,0700","1545,37",1096,03/12/2007,2007.0,3528809.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379509,Nobre Seguradora do Brasil S.A,15414002899200612,165721551655,640739,07/05/2013,23/07/2016,23/07/2016,ANTONIO RINALDI,***17416068,Bento Gonçalves,...,"178301,03","14995,14","0,0841","8997,08",75356.0,31/05/2013,2013.0,4302105.0,NaN,NaN
379510,Nobre Seguradora do Brasil S.A,15414002899200612,165721554977,640740,08/05/2013,23/07/2016,23/07/2016,AZIR TOMASI,***19478087,Bento Gonçalves,...,"45195,48","1970,52","0,0436","1182,31",75613.0,04/06/2013,2013.0,4302105.0,NaN,NaN
379511,Nobre Seguradora do Brasil S.A,15414002899200612,165771586336,640741,11/06/2013,23/07/2016,23/07/2016,ARDUINO PAVAN,***19729004,Bento Gonçalves,...,35600,3382,"0,0950","2029,2",76564.0,19/06/2013,2013.0,4302105.0,"5133,59",GRANIZO
379512,Nobre Seguradora do Brasil S.A,15414001852200712,165671524628,640742,26/03/2013,23/07/2016,23/07/2016,EDSON ANDRE CROSATTI,***22059960,Apucarana,...,44424,"4837,78","0,1089","3386,45",73193.0,09/05/2013,2013.0,4101408.0,"35539,2",GEADA


In [ ]:

class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space

    def select_action(self, observation):
        return self.action_space.sample()


def run_random_agent(num_episodes, budget):
            # Initialize the InsurEnv environment
    env = gym.make(
        "InsurEnv-v0",
        data=data,
        index_column='index_column',
        state_columns='state_columns',
        value_column='value_column',
        budget=budget,
    )
    episode_results = []
    agent = RandomAgent(env.action_space)

    for episode in range(num_episodes):
        observation, _ = env.reset()
        done = False
        acts =[]
        while not done:
            action = agent.select_action(observation)
            observation, reward, done, truncated, _ = env.step(action)
            # Optionally, render the environment
            # env.render()
        print(f"Episode {episode + 1} finished")
        df = env.results.copy(deep = True)
        df['episode'] = episode + 1
        episode_results.append(df)
    env.close()
    return episode_results


def run_bootstrap_random_agent(budget, k=10):
    bootstrap_results = []
    num_episodes = 100

    for i in range(k):
        results = run_random_agent(num_episodes=num_episodes, budget = budget)
        results  = pd.concat(results, axis =0)
        results['boostrap_batch'] = i +1
        bootstrap_results.append(results)

    return pd.concat(bootstrap_results, axis =0)  
     
def run_budget_boostrap_random_agent():
    budget_list = [1_000_000, 5_000_000, 10_000_000, 15_000_000]
    budget_results = []
    for b in budget_list:
        results = run_bootstrap_random_agent(budget = b, k = 10 )
        results = pd.concat(results, axis = 0)
        budget_results.append(results)
    return pd.concat(budget_results, axis = 0 )



# Run the random agent
results = run_budget_boostrap_random_agent()
results

NameError: name 'data' is not defined

In [ ]:
            # Initialize the InsurEnv environment
    env = gym.make(
        "InsurEnv-v0",
        data=data,
        index_column='index_column',
        state_columns='state_columns',
        value_column='value_column',
        budget=budget,
    )
    episode_results = []
    agent = RandomAgent(env.action_space)

    for episode in range(num_episodes):
        observation, _ = env.reset()
        done = False
        acts =[]
        while not done:
            action = agent.select_action(observation)
            observation, reward, done, truncated, _ = env.step(action)
            # Optionally, render the environment
            # env.render()
        print(f"Episode {episode + 1} finished")
        df = env.results.copy(deep = True)
        df['episode'] = episode + 1
        episode_results.append(df)
    env.close()
    return episode_results


def run_bootstrap_random_agent(budget, k=10):
    bootstrap_results = []
    num_episodes = 100

    for i in range(k):
        results = run_random_agent(num_episodes=num_episodes, budget = budget)
        results  = pd.concat(results, axis =0)
        results['boostrap_batch'] = i +1
        bootstrap_results.append(results)

    return pd.concat(bootstrap_results, axis =0)  
     
def run_budget_boostrap_random_agent():
    budget_list = [1_000_000, 5_000_000, 10_000_000, 15_000_000]
    budget_results = []
    for b in budget_list:
        results = run_bootstrap_random_agent(budget = b, k = 10 )
        results = pd.concat(results, axis = 0)
        budget_results.append(results)
    return pd.concat(budget_results, axis = 0 )



# Run the random agent
results = run_budget_boostrap_random_agent()
results

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(num_episodes), results.groupby('episode')['cum_rewards'].last(), label="Rewards")
plt.xlabel("Steps")
plt.ylabel("Rewards")
plt.title("Rewards Over Time")
plt.legend()
plt.show()

In [ ]:
results.groupby('episode')['cum_rewards'].last().hist()

In [ ]:
!pip install tensorflow-io

In [ ]:
import gymnasium as gym


import pandas as pd

import matplotlib.pyplot as plt
import numpy as np
from gym_insurance.envs.insurenv import InsurEnv

size = 2000

# Create a sample dataset
data = pd.DataFrame({
    'index_column': range(size),
    'value_column': np.random.rand(size) * 1000,
    'feature_1': np.random.rand(size),
    'feature_2': np.random.rand(size)
})

from stable_baselines3 import A2C

env = InsurEnv(
data=data,
index_column='index_column',
value_column='value_column',
state_columns=['feature_1', 'feature_2'],
budget=2_000
)
model = A2C("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10_000)

vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)



In [ ]:
import gym

In [ ]:
gym.__version__

In [ ]:
!pip install gymnasium==0.29.1